## Notebook summary

| Item | Details |
| --- | --- |
| Purpose | DenseNet-121 YOLO-ROI paired fine-tune from notebook 01 |
| Model | DenseNet-121 (from notebook 01 best checkpoint) |
| Input | Paired published crop + YOLO square ROI (alternates 50/50) |
| Training | Single-stage fine-tune, 5 epochs, AdamW + CosineAnnealingLR |
| Loss | Plain CrossEntropyLoss |
| Selection | QWK only |
| Outputs | best_model.pth, last_model.pth, history.csv, metadata.json |
| Status | Compact paired-view fine-tune (matches working baseline pattern) |


## Detailed config

### Identity

| Item | Value |
| --- | --- |
| Purpose | Fine-tune DenseNet-121 from notebook 01 on paired published+YOLO views |
| Base checkpoint | notebook 01 best_model.pth (original optimized) |
| Output dir | `/content/drive/MyDrive/Models/densenet121_yolo_roi/<TIMESTAMP>/` |

### Dataset

| Item | Value |
| --- | --- |
| Classes | 5 KL grades (0-4) |
| Published root | `/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted/KneeXrayData/ClsKLData/kneeKL224` |
| ROI root | `/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/densenet121_yolo_square_roi_trainvaltest_v2` |
| Input size | 384x384 |
| Augmentation | OpenCV CLAHE -> SquarePad -> PIL -> HFlip(p=0.5) -> Rotation(5) -> ColorJitter(0.08,0.08) -> Resize(384) -> RandomErasing(0.10) -> ImageNet norm |

### Training

| Item | Value |
| --- | --- |
| Epochs | 5 |
| LR | 1e-5 |
| Weight decay | 1e-3 |
| Batch size | 48 |
| Num workers | 2 |
| Scheduler | CosineAnnealingLR (stepped once per epoch) |
| Sampler | WeightedRandomSampler (inverse-class-frequency, power=1.0) |
| Loss | CrossEntropyLoss |
| Val views | Evaluated on both published and YOLO ROI views each epoch |

### Selection

| Item | Value |
| --- | --- |
| Robust selection | 0.5 * (published_selection + roi_selection) |
| Checkpoints | best_model.pth (max robust_selection), last_model.pth (every epoch) |


# DenseNet-121 YOLO-ROI Paired Fine-Tune

Fine-tune the notebook-01 DenseNet-121 checkpoint on paired published+YOLO views.
5 epochs, CosineAnnealingLR, WeightedRandomSampler, single-stage.
Matches the working paired-view baseline pattern.


## 0. Setup


In [3]:
!pip -q install 'timm>=1.0' 'h5py>=3.9'


In [4]:
from google.colab import drive
drive.mount('/content/drive')

import json
import random
from datetime import datetime, timezone
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (average_precision_score, classification_report, cohen_kappa_score, confusion_matrix, mean_absolute_error, precision_recall_fscore_support, roc_auc_score)
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from tqdm.auto import tqdm


Mounted at /content/drive


## 1. Configuration


In [5]:
# ─── Paths ───────────────────────────────────────────────────────────────────
PUBLISHED_ROOT = Path(
    '/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/'
    'extracted/KneeXrayData/ClsKLData/kneeKL224'
)
ROI_ROOT = Path(
    '/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/'
    'derived/densenet121_yolo_square_roi_trainvaltest_v2'
)
# Base checkpoint = notebook 01's best_model.pth (auto-discovers the latest
# completed run under the optimized_original directory; fails fast if none exist).
BASE_CHECKPOINT_ROOT = Path('/content/drive/MyDrive/Models/densenet121_optimized_original')
candidates = sorted(
    BASE_CHECKPOINT_ROOT.glob('*/best_model.pth'),
    key=lambda p: p.stat().st_mtime, reverse=True
)
if not candidates:
    raise FileNotFoundError(
        f'No best_model.pth under {BASE_CHECKPOINT_ROOT}. '
        'Run notebook 01 first to produce a base checkpoint.'
    )
BASE_CHECKPOINT = candidates[0]
print(f'Base checkpoint (latest by mtime): {BASE_CHECKPOINT}')

# ─── Test split paths ─────────────────────────────────────────────────────────
# The test split is a completely held-out set — NOT used during training.
# ROI_TEST_ROOT: your own YOLO-cropped test images  (same v2 folder, /test subfolder)
# PUB_TEST_ROOT:  published-view test images  (same kneeKL224 folder, /test subfolder)
ROI_TEST_ROOT = ROI_ROOT / 'test'
PUB_TEST_ROOT = PUBLISHED_ROOT / 'test'
CASES_PER_GRADE = 5   # how many correct/incorrect Grad-CAM samples to show per grade

# ─── Training ─────────────────────────────────────────────────────────────────
SEED = 42
INPUT_SIZE = 384
IS_A100 = torch.cuda.is_available() and "A100" in torch.cuda.get_device_name(0)
BATCH_SIZE = 32 if IS_A100 else 16
NUM_WORKERS = 8 if IS_A100 else 2
EPOCHS = 5
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-3
ALTERNATE_VIEW_PROBABILITY = 0.50  # 50 % chance to use YOLO ROI instead of published crop

# ─── Derived ────────────────────────────────────────────────────────────────
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime('%Y-%m-%d_%H-%M-%S_%f_UTC')
RUN_DIR = Path('/content/drive/MyDrive/Models/densenet121_yolo_roi') / RUN_TIMESTAMP

for p in (PUBLISHED_ROOT, ROI_ROOT, BASE_CHECKPOINT):
    if not p.exists():
        raise FileNotFoundError(p)
RUN_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print(f'Base checkpoint: {BASE_CHECKPOINT}')
print(f'Epochs: {EPOCHS}  LR: {LEARNING_RATE}  Batch size: {BATCH_SIZE}  Workers: {NUM_WORKERS}')
print(f'Scheduler: CosineAnnealingLR (stepped once per epoch)')
print(f'Alternate view probability: {ALTERNATE_VIEW_PROBABILITY}')


Base checkpoint (latest by mtime): /content/drive/MyDrive/Models/densenet121_optimized_original/2026-08-20_15-15-22_677034_UTC/best_model.pth
Device: cuda
Base checkpoint: /content/drive/MyDrive/Models/densenet121_optimized_original/2026-08-20_15-15-22_677034_UTC/best_model.pth
Epochs: 5  LR: 1e-05  Batch size: 16  Workers: 2
Scheduler: CosineAnnealingLR (stepped once per epoch)
Alternate view probability: 0.5


## 2. Build paired published / YOLO records


In [6]:
rows = []
for split in ('train', 'val'):
    for grade in range(5):
        for pub in sorted((PUBLISHED_ROOT / split / str(grade)).glob('*.png')):
            roi = ROI_ROOT / split / str(grade) / pub.name
            if not roi.is_file():
                raise FileNotFoundError(f'Missing paired ROI: {roi}')
            rows.append({'split': split, 'grade': grade,
                         'published_path': str(pub), 'roi_path': str(roi)})

frame = pd.DataFrame(rows)
print(frame.groupby(['split', 'grade']).size().unstack(fill_value=0))

train_frame = frame[frame.split == 'train'].reset_index(drop=True)
val_frame   = frame[frame.split == 'val'  ].reset_index(drop=True)

counts = np.bincount(train_frame.grade.to_numpy(), minlength=5)
weights = (1.0 / counts)[train_frame.grade.to_numpy()]
sampler = WeightedRandomSampler(
    torch.as_tensor(weights, dtype=torch.double), len(weights), replacement=True
)
print(f'\nClass counts: {dict(enumerate(counts))}')


grade     0     1     2    3    4
split                            
train  2286  1046  1516  757  173
val     328   153   212  106   27

Class counts: {0: np.int64(2286), 1: np.int64(1046), 2: np.int64(1516), 3: np.int64(757), 4: np.int64(173)}


## 3. Preprocessing, Dataset & Model


In [7]:
class OpenCVCLAHE:
    def __call__(self, image_rgb):
        lab = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        l = cv2.createCLAHE(clipLimit=1.25, tileGridSize=(8, 8)).apply(l)
        return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)

class SquarePad:
    def __call__(self, image_rgb):
        h, w = image_rgb.shape[:2]
        side = max(h, w)
        top = (side - h) // 2
        left = (side - w) // 2
        return cv2.copyMakeBorder(
            image_rgb, top, side - h - top, left, side - w - left,
            cv2.BORDER_CONSTANT, value=(0, 0, 0))

normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

train_transform = transforms.Compose([
    OpenCVCLAHE(), SquarePad(), transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.50),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.08, contrast=0.08),
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.10, scale=(0.02, 0.05), ratio=(0.5, 2.0), value=0),
    normalize,
])

val_transform = transforms.Compose([
    OpenCVCLAHE(), SquarePad(), transforms.ToPILImage(),
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    normalize,
])

class PairedDataset(Dataset):
    def __init__(self, data, transform, alternate_probability):
        self.data = data.reset_index(drop=True)
        self.transform = transform
        self.alternate_probability = alternate_probability
        self.labels = self.data.grade.astype(int).tolist()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        use_roi = (self.alternate_probability > 0 and
                   random.random() < self.alternate_probability)
        img = cv2.imread(row.roi_path if use_roi else row.published_path)
        if img is None:
            raise IOError(f'Cannot read image at index {index}')
        return self.transform(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)), int(row.grade)


# ─── Test datasets (your own YOLO crops) ────────────────────────────────────
# ROITestDataset  — reads from ROI_TEST_ROOT (your own YOLO-cropped test images)
# PublishedTestDataset — reads from PUB_TEST_ROOT (published full-view test images)
# Both produce the same val_transform preprocessing as the training/val splits.

class ROITestDataset(Dataset):
    """Single-view dataset: reads YOLO-cropped test images from ROI_TEST_ROOT."""
    def __init__(self, root, transform):
        rows = []
        for grade in range(5):
            for path in sorted((root / str(grade)).glob('*.png')):
                rows.append({'path': str(path), 'true_grade': grade})
        self.frame = pd.DataFrame(rows)
        self.transform = transform

    def __len__(self): return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        img = cv2.imread(row['path'])
        if img is None: raise IOError(f'Cannot read: {row["path"]}')
        tensor = self.transform(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        return tensor, int(row['true_grade']), row['path']


class PublishedTestDataset(Dataset):
    """Single-view dataset: reads published full-view test images from PUB_TEST_ROOT."""
    def __init__(self, root, transform):
        rows = []
        for grade in range(5):
            for path in sorted((root / str(grade)).glob('*.png')):
                rows.append({'path': str(path), 'true_grade': grade})
        self.frame = pd.DataFrame(rows)
        self.transform = transform

    def __len__(self): return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        img = cv2.imread(row['path'])
        if img is None: raise IOError(f'Cannot read: {row["path"]}')
        tensor = self.transform(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        return tensor, int(row['true_grade']), row['path']


class DenseNet121Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            'densenet121', pretrained=False, num_classes=5, drop_rate=0.20)

    @property
    def gradcam_target_layer(self):
        """Hook point for Grad-CAM: the last conv layer of DenseNet-121."""
        return self.backbone.features.norm5

    def forward(self, images):
        return self.backbone(images)


class DenseNetGradCAM:
    """Grad-CAM using features.norm5 as the target conv layer."""
    def __init__(self, model):
        self.model = model
        self.activations = None
        self.gradients  = None
        self.handle = model.gradcam_target_layer.register_forward_hook(self._save_act)
        model.gradcam_target_layer.register_full_backward_hook(self._save_grad)

    def _save_act(self, module, inputs, output):
        self.activations = output.detach()

    def _save_grad(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def __call__(self, tensor, class_idx):
        self.model.zero_grad(set_to_none=True)
        logits = self.model(tensor)
        logits[0, class_idx].backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * self.activations).sum(dim=1)).squeeze()
        cam = cam.cpu().numpy()
        cam -= cam.min();  cam /= max(cam.max(), 1e-8)
        return cam

    def close(self):
        self.handle.remove()


model = DenseNet121Model().to(DEVICE)
checkpoint = torch.load(BASE_CHECKPOINT, map_location='cpu', weights_only=False)
if 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'], strict=True)
print(f'Loaded base checkpoint: {BASE_CHECKPOINT}')
total = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total:,}')

train_loader = DataLoader(
    PairedDataset(train_frame, train_transform, ALTERNATE_VIEW_PROBABILITY),
    batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=True,
)
val_pub_loader = DataLoader(
    PairedDataset(val_frame, val_transform, 0.0),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
val_roi_loader = DataLoader(
    PairedDataset(val_frame, val_transform, 1.0),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
# ─── Test dataloaders (held-out, no augmentation) ────────────────────────────
test_roi_loader = DataLoader(
    ROITestDataset(ROI_TEST_ROOT, val_transform),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
test_pub_loader = DataLoader(
    PublishedTestDataset(PUB_TEST_ROOT, val_transform),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
print(f'Train batches: {len(train_loader)}')
print(f'Val batches  (published): {len(val_pub_loader)}')
print(f'Val batches  (ROI):       {len(val_roi_loader)}')
print(f'Test batches (YOLO-ROI):  {len(test_roi_loader)}')
print(f'Test batches (Published): {len(test_pub_loader)}')


Loaded base checkpoint: /content/drive/MyDrive/Models/densenet121_optimized_original/2026-08-20_15-15-22_677034_UTC/best_model.pth
Total parameters: 6,958,981  Trainable: 6,958,981 (all layers unfrozen)
Train batches: 362
Val batches (published): 52
Val batches (ROI):       52


## 4. Training Loop


In [8]:
# ─── Plain CrossEntropyLoss — no mixup, no ordinal tricks ─────────────
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch.nn.functional as F
from sklearn.metrics import (average_precision_score, cohen_kappa_score,
                             confusion_matrix, precision_recall_fscore_support,
                             roc_auc_score)
from sklearn.preprocessing import label_binarize

GRADE_NAMES = ['0 - Normal', '1 - Doubtful', '2 - Mild', '3 - Moderate', '4 - Severe']
NUM_CLASSES = 5

def evaluate_published(loader, tag='Pub'):
    model.eval()
    all_labels, all_preds, all_probas = [], [], []
    with torch.inference_mode():
        for images, labels in tqdm(loader, desc=f'[{tag}]'):
            images = images.to(DEVICE, non_blocking=True)
            logits = model(images).float()
            probas = F.softmax(logits, dim=1).cpu().numpy()
            all_labels.extend(labels.numpy())
            all_preds.extend(logits.argmax(dim=1).cpu().numpy())
            all_probas.extend(probas)

    y_true = np.asarray(all_labels).astype(int)
    y_pred = np.asarray(all_preds).astype(int)
    y_proba = np.asarray(all_probas)
    y_onehot = label_binarize(y_true, classes=range(NUM_CLASSES))

    # ── Aggregate metrics ──
    qwk = float(cohen_kappa_score(y_true, y_pred, weights='quadratic'))
    macro_f1, macro_pr, macro_re, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0)
    macro_ap = float(average_precision_score(y_onehot, y_proba, average='macro'))

    # Per-class AP & ROC-AUC
    per_class_ap, per_class_auc = [], []
    for c in range(NUM_CLASSES):
        ap_c = float(average_precision_score(y_onehot[:, c], y_proba[:, c]))
        per_class_ap.append(ap_c)
        try:
            auc_c = float(roc_auc_score(y_onehot[:, c], y_proba[:, c]))
        except ValueError:
            auc_c = float('nan')
        per_class_auc.append(auc_c)
    mAP = float(np.nanmean(per_class_ap))
    mAUC = float(np.nanmean([x for x in per_class_auc if not np.isnan(x)]))

    # Per-class P/R/F1/support
    p_c, r_c, f_c, s_c = precision_recall_fscore_support(
        y_true, y_pred, labels=range(NUM_CLASSES), zero_division=0)

    return {
        'accuracy': float(np.mean(y_true == y_pred)),
        'qwk': qwk,
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'off1_acc': float(np.mean(np.abs(y_true - y_pred) <= 1)),
        'macro_f1': float(macro_f1),
        'macro_pr': float(macro_pr),
        'macro_re': float(macro_re),
        'macro_ap': macro_ap,
        'mAP': mAP,
        'mAUC': mAUC,
        'confusion_matrix': confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES)).tolist(),
        'per_class': {
            'precision': [float(x) for x in p_c],
            'recall':    [float(x) for x in r_c],
            'f1':        [float(x) for x in f_c],
            'support':   [int(x)   for x in s_c],
            'ap':        per_class_ap,
            'auc':       per_class_auc,
        }
    }


def evaluate_roi(loader):
    return evaluate_published(loader, tag='ROI')


# ─── Training loop: fine-tune from notebook 01 checkpoint, alternating views ──
LR_STAGE_HEAD = LEARNING_RATE
optimizer = torch.optim.AdamW(model.parameters(), lr=LR_STAGE_HEAD, weight_decay=WEIGHT_DECAY)

# Cosine schedule across the full EPOCHS budget
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')

criterion = nn.CrossEntropyLoss()

best_score = -float('inf')
history = []

last_checkpoint_path = RUN_DIR / 'last_model.pth'
best_checkpoint_path = RUN_DIR / 'best_model.pth'

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss, epoch_correct, epoch_total = 0.0, 0, 0
    progress = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}')
    for images, labels in progress:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
            logits = model(images)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item() * labels.size(0)
        epoch_correct += (logits.argmax(dim=1) == labels).sum().item()
        epoch_total += labels.size(0)
        progress.set_postfix(loss=f'{epoch_loss / epoch_total:.4f}', acc=f'{epoch_correct / epoch_total:.4f}')

    scheduler.step()
    train_loss = epoch_loss / max(epoch_total, 1)
    train_acc = epoch_correct / max(epoch_total, 1)

    pub_metrics = evaluate_published(val_pub_loader)
    roi_metrics = evaluate_roi(val_roi_loader)
    robust_selection = 0.5 * (pub_metrics['qwk'] + roi_metrics['qwk'])

    history.append({
        'epoch': epoch,
        'lr': optimizer.param_groups[0]['lr'],
        'train_loss': train_loss,
        'train_acc': train_acc,
        'pub_qwk': pub_metrics['qwk'],
        'pub_acc': pub_metrics['accuracy'],
        'pub_macro_f1': pub_metrics['macro_f1'],
        'pub_mAP': pub_metrics['mAP'],
        'roi_qwk': roi_metrics['qwk'],
        'roi_acc': roi_metrics['accuracy'],
        'roi_macro_f1': roi_metrics['macro_f1'],
        'roi_mAP': roi_metrics['mAP'],
        'robust_selection': robust_selection,
    })

    print(
        f'Epoch {epoch}: loss={train_loss:.4f} acc={train_acc:.4f} | '
        f'pub_qwk={pub_metrics["qwk"]:.4f} roi_qwk={roi_metrics["qwk"]:.4f} | '
        f'robust={robust_selection:.4f}'
    )

    payload = {
        'model_state_dict': model.state_dict(),
        'selection': robust_selection,
        'pub_metrics': pub_metrics,
        'roi_metrics': roi_metrics,
    }
    torch.save(payload, last_checkpoint_path)

    if robust_selection > best_score:
        best_score = robust_selection
        torch.save(payload, best_checkpoint_path)
        print(f'  -> New best! Robust={best_score:.4f}')


In [9]:
# ─── Training loop: fine-tune from notebook 01 checkpoint, alternating views ──
LR_STAGE_HEAD = LEARNING_RATE
optimizer = torch.optim.AdamW(model.parameters(), lr=LR_STAGE_HEAD, weight_decay=WEIGHT_DECAY)

# Cosine schedule across the full EPOCHS budget
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')

criterion = nn.CrossEntropyLoss()

best_score = -float('inf')
history = []

last_checkpoint_path = RUN_DIR / 'last_model.pth'
best_checkpoint_path = RUN_DIR / 'best_model.pth'

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss, epoch_correct, epoch_total = 0.0, 0, 0
    progress = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}')
    for images, labels in progress:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
            logits = model(images)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item() * labels.size(0)
        epoch_correct += (logits.argmax(dim=1) == labels).sum().item()
        epoch_total += labels.size(0)
        progress.set_postfix(loss=f'{epoch_loss / epoch_total:.4f}', acc=f'{epoch_correct / epoch_total:.4f}')

    scheduler.step()
    train_loss = epoch_loss / max(epoch_total, 1)
    train_acc = epoch_correct / max(epoch_total, 1)

    pub_metrics = evaluate_published(val_pub_loader)
    roi_metrics = evaluate_roi(val_roi_loader)
    # Robust selection = mean of the two QWKs (eval-views must agree on improvements)
    robust_selection = 0.5 * (pub_metrics['qwk'] + roi_metrics['qwk'])

    history.append({
        'epoch': epoch,
        'lr': optimizer.param_groups[0]['lr'],
        'train_loss': train_loss,
        'train_acc': train_acc,
        'pub_qwk': pub_metrics['qwk'],
        'pub_acc': pub_metrics['accuracy'],
        'pub_macro_f1': pub_metrics['macro_f1'],
        'roi_qwk': roi_metrics['qwk'],
        'roi_acc': roi_metrics['accuracy'],
        'roi_macro_f1': roi_metrics['macro_f1'],
        'robust_selection': robust_selection,
    })

    print(
        f'Epoch {epoch}: loss={train_loss:.4f} acc={train_acc:.4f} | '
        f'pub_qwk={pub_metrics["qwk"]:.4f} roi_qwk={roi_metrics["qwk"]:.4f} | '
        f'robust={robust_selection:.4f}'
    )

    payload = {
        'model_state_dict': model.state_dict(),
        'selection': robust_selection,
        'pub_metrics': pub_metrics,
        'roi_metrics': roi_metrics,
    }
    torch.save(payload, last_checkpoint_path)

    if robust_selection > best_score:
        best_score = robust_selection
        torch.save(payload, best_checkpoint_path)
        print(f'  -> New best! Robust={best_score:.4f}')


Epoch 1/5:   0%|          | 0/362 [00:00<?, ?it/s]

[Pub]:   0%|          | 0/52 [00:00<?, ?it/s]

[ROI]:   0%|          | 0/52 [00:00<?, ?it/s]

Epoch 1: loss=1.0024 acc=0.5628 | pub_qwk=0.7433 roi_qwk=0.6776 | robust=0.7104
  -> New best! Robust=0.7104


Epoch 2/5:   0%|          | 0/362 [00:00<?, ?it/s]

...

Epoch 4: loss=0.8079 acc=0.6435 | pub_qwk=0.7517 roi_qwk=0.7087 | robust=0.7302


Epoch 5/5:   0%|          | 0/362 [00:00<?, ?it/s]

[Pub]:   0%|          | 0/52 [00:00<?, ?it/s]

[ROI]:   0%|          | 0/52 [00:00<?, ?it/s]

Epoch 5: loss=0.8099 acc=0.6331 | pub_qwk=0.7758 roi_qwk=0.7116 | robust=0.7437
  -> New best! Robust=0.7437


In [10]:
# ─── Final evaluation: load best checkpoint and report comprehensive metrics ──
best_ckpt = torch.load(best_checkpoint_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(best_ckpt["model_state_dict"])
print(f"Loaded best checkpoint (robust={best_ckpt['selection']:.4f})\n")

pub_m = evaluate_published(val_pub_loader)
roi_m = evaluate_roi(val_roi_loader)

# ══════════════════════════════════════════════════════════════════════════════
# 1. AGGREGATE METRICS TABLE
# ══════════════════════════════════════════════════════════════════════════════
agg_rows = [
    ('Accuracy',            pub_m['accuracy'],   roi_m['accuracy']),
    ('QWK',                 pub_m['qwk'],        roi_m['qwk']),
    ('MAE',                 pub_m['mae'],        roi_m['mae']),
    ('Off-by-1 Accuracy',   pub_m['off1_acc'],   roi_m['off1_acc']),
    ('Macro Precision',     pub_m['macro_pr'],   roi_m['macro_pr']),
    ('Macro Recall',        pub_m['macro_re'],   roi_m['macro_re']),
    ('Macro F1',            pub_m['macro_f1'],   roi_m['macro_f1']),
    ('Macro AP',            pub_m['macro_ap'],   roi_m['macro_ap']),
    ('mAP',                 pub_m['mAP'],        roi_m['mAP']),
    ('mAUC',                pub_m['mAUC'],       roi_m['mAUC']),
]
print("=" * 74)
print(f"{'Aggregate Metrics':30s}  {'Published':>12s}  {'YOLO-ROI':>12s}")
print("-" * 74)
for name, pv, rv in agg_rows:
    print(f"{name:30s}  {pv:12.4f}  {rv:12.4f}")
print("=" * 74)

# ══════════════════════════════════════════════════════════════════════════════
# 2. PER-CLASS METRICS TABLE  (P / R / F1 / AP / AUC / Support)
# ══════════════════════════════════════════════════════════════════════════════
header = f"{'Grade':<22s}  {'Prec':>6s}  {'Rec':>6s}  {'F1':>6s}  {'AP':>6s}  {'AUC':>6s}  {'N':>5s}"
sep    = "=" * 72

print(f"\n{'Published View — Per-Class Metrics':^72}")
print(sep)
print(f"{'':22s}  {'Prec':>6s}  {'Rec':>6s}  {'F1':>6s}  {'AP':>6s}  {'AUC':>6s}  {'N':>5s}")
print(sep)
for i, name in enumerate(GRADE_NAMES):
    pc = pub_m['per_class']
    print(f"{name:<22s}  "
          f"{pc['precision'][i]:6.4f}  {pc['recall'][i]:6.4f}  {pc['f1'][i]:6.4f}  "
          f"{pc['ap'][i]:6.4f}  {pc['auc'][i]:6.4f}  {pc['support'][i]:5d}")
print(sep)

print(f"\n{'YOLO-ROI View — Per-Class Metrics':^72}")
print(sep)
print(f"{'':22s}  {'Prec':>6s}  {'Rec':>6s}  {'F1':>6s}  {'AP':>6s}  {'AUC':>6s}  {'N':>5s}")
print(sep)
for i, name in enumerate(GRADE_NAMES):
    pc = roi_m['per_class']
    print(f"{name:<22s}  "
          f"{pc['precision'][i]:6.4f}  {pc['recall'][i]:6.4f}  {pc['f1'][i]:6.4f}  "
          f"{pc['ap'][i]:6.4f}  {pc['auc'][i]:6.4f}  {pc['support'][i]:5d}")
print(sep)

# ══════════════════════════════════════════════════════════════════════════════
# 3. CONFUSION MATRICES  (Published | YOLO-ROI)
# ══════════════════════════════════════════════════════════════════════════════
def print_cm(cm, title):
    print(f"\n{title}")
    col_w = 7
    header_row = f"{'':>{col_w}s}" + ''.join(f"{i:>{col_w}d}" for i in range(NUM_CLASSES))
    print(header_row)
    for i, row in enumerate(cm):
        row_str = f"{i:>{col_w}d}" + ''.join(f"{v:>{col_w}d}" for v in row)
        print(row_str)

print_cm(pub_m['confusion_matrix'], 'Confusion Matrix — Published View:')
print_cm(roi_m['confusion_matrix'], 'Confusion Matrix — YOLO-ROI View:')

# ══════════════════════════════════════════════════════════════════════════════
# 4. SAVE FULL METRICS JSON
# ══════════════════════════════════════════════════════════════════════════════
final_metrics = {
    'best_robust_selection': float(best_score),
    'published': pub_m,
    'yolo_roi': roi_m,
}
with open(RUN_DIR / 'final_metrics.json', 'w') as f:
    json.dump(final_metrics, f, indent=2)

print(f"\nSaved: {RUN_DIR / 'final_metrics.json'}")


In [ ]:
# ─── 5. Locked TEST evaluation + Grad-CAM review ─────────────────────────────────
# This cell evaluates the best checkpoint on the HELD-OUT test split once.
# It does NOT influence checkpoint selection — only used for final reporting.
# Grad-CAM audit helps distinguish joint-space evidence from shortcut patterns.

best_ckpt = torch.load(best_checkpoint_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(best_ckpt['model_state_dict'], strict=True)
model.eval()
print(f"Loaded: {best_checkpoint_path}  (robust={best_ckpt['selection']:.4f})\n")

# ── Helper: evaluate a single-view test loader ──────────────────────────────────
def evaluate_test(loader, tag):
    all_labels, all_preds, all_probs = [], [], []
    with torch.inference_mode():
        for images, labels, _ in tqdm(loader, desc=f'Test [{tag}]'):
            probs = F.softmax(model(images.to(DEVICE, non_blocking=True).float()), dim=1)
            all_labels.extend(labels.cpu().numpy().tolist())
            all_preds.extend(probs.argmax(dim=1).cpu().numpy().tolist())
            all_probs.extend(probs.cpu().numpy())
    y_true = np.asarray(all_labels, dtype=int)
    y_pred = np.asarray(all_preds, dtype=int)
    y_prob = np.asarray(all_probs, dtype=float)
    return {
        'accuracy': float(np.mean(y_true == y_pred)),
        'qwk':      float(cohen_kappa_score(y_true, y_pred, weights='quadratic')),
        'mae':      float(mean_absolute_error(y_true, y_pred)),
        'off1_acc': float(np.mean(np.abs(y_true - y_pred) <= 1)),
        'macro_pr': float(precision_recall_fscore_support(
            y_true, y_pred, average='macro', zero_division=0)[0]),
        'macro_re': float(precision_recall_fscore_support(
            y_true, y_pred, average='macro', zero_division=0)[1]),
        'macro_f1': float(precision_recall_fscore_support(
            y_true, y_pred, average='macro', zero_division=0)[2]),
        'macro_ap': float(average_precision_score(np.eye(5)[y_true], y_prob, average='macro')),
        'mAUC':     float(roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro')),
        'y_true': y_true, 'y_pred': y_pred, 'y_prob': y_prob,
    }

test_roi_m = evaluate_test(test_roi_loader, 'YOLO-ROI')
test_pub_m = evaluate_test(test_pub_loader, 'Published')

agg_rows = [
    ('Accuracy',         test_pub_m['accuracy'],  test_roi_m['accuracy']),
    ('QWK',              test_pub_m['qwk'],      test_roi_m['qwk']),
    ('MAE',              test_pub_m['mae'],       test_roi_m['mae']),
    ('Off-by-1 Acc',     test_pub_m['off1_acc'],  test_roi_m['off1_acc']),
    ('Macro F1',         test_pub_m['macro_f1'],  test_roi_m['macro_f1']),
    ('Macro AP',         test_pub_m['macro_ap'],  test_roi_m['macro_ap']),
    ('mAUC',             test_pub_m['mAUC'],      test_roi_m['mAUC']),
]
print('=' * 68)
print(f"{'TEST Split Metrics':30s}  {'Published':>12s}  {'YOLO-ROI':>12s}")
print('-' * 68)
for name, pv, rv in agg_rows:
    print(f"{name:30s}  {pv:12.4f}  {rv:12.4f}")
print('=' * 68)

# ── Grad-CAM audit ─────────────────────────────────────────────────────────────
CAM_DIR = RUN_DIR / 'gradcam_test'
CAM_DIR.mkdir(exist_ok=True)
correct_dir = CAM_DIR / 'correct'
incorrect_dir = CAM_DIR / 'incorrect'
(correct_dir / 'roi').mkdir(parents=True, exist_ok=True)
(correct_dir / 'pub').mkdir(parents=True, exist_ok=True)
(incorrect_dir / 'roi').mkdir(parents=True, exist_ok=True)
(incorrect_dir / 'pub').mkdir(parents=True, exist_ok=True)

def draw_cam(img_tensor, cam, stem, view_tag, true_lbl, pred_lbl, save_path):
    img = img_tensor.cpu().numpy().transpose(1, 2, 0)
    img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img = np.clip(img, 0, 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(img)
    axes[0].set_title(f'{stem} [{view_tag}]\nTrue={true_lbl}  Pred={pred_lbl}')
    axes[0].axis('off')
    hm = axes[1].imshow(img)
    axes[1].imshow(cam, cmap='jet', alpha=0.55)
    axes[1].set_title('Grad-CAM overlay')
    axes[1].axis('off')
    fig.colorbar(hm, ax=axes[1], fraction=0.046)
    plt.tight_layout()
    fig.savefig(save_path, bbox_inches='tight', dpi=120)
    plt.close(fig)

def collect_cam_samples(loader, gc, view_tag, correct_sub, incorrect_sub):
    correct_list, incorrect_list = [], []
    with torch.inference_mode():
        for tensors, labels, paths in tqdm(loader, desc=f'CAM [{view_tag}]'):
            probs = F.softmax(model(tensors.to(DEVICE).float()), dim=1)
            preds = probs.argmax(dim=1).cpu().numpy()
            for j in range(tensors.size(0)):
                cam = gc(tensors[j:j+1].to(DEVICE), int(preds[j]))
                entry = (tensors[j], cam, Path(paths[j]).stem, int(labels[j]), int(preds[j]))
                if preds[j] == labels[j].item() and len(correct_list) < 5 * CASES_PER_GRADE:
                    correct_list.append(entry)
                elif preds[j] != labels[j].item() and len(incorrect_list) < 5 * CASES_PER_GRADE:
                    incorrect_list.append(entry)
    for idx, (tensor, cam, stem, true_g, pred_g) in enumerate(correct_list):
        draw_cam(tensor, cam, f'{stem}_{idx}', view_tag, true_g, pred_g,
                 correct_sub / f'{stem}_{idx}.png')
    for idx, (tensor, cam, stem, true_g, pred_g) in enumerate(incorrect_list):
        draw_cam(tensor, cam, f'{stem}_{idx}', view_tag, true_g, pred_g,
                 incorrect_sub / f'{stem}_{idx}.png')

gc_roi = DenseNetGradCAM(model)
gc_pub = DenseNetGradCAM(model)
collect_cam_samples(test_roi_loader, gc_roi, 'ROI', correct_dir / 'roi', incorrect_dir / 'roi')
collect_cam_samples(test_pub_loader, gc_pub, 'Pub', correct_dir / 'pub', incorrect_dir / 'pub')
gc_roi.close();  gc_pub.close()

print(f'\nGrad-CAM saved: {CAM_DIR}')
print('correct/ → roi/, pub/    incorrect/ → roi/, pub/')

# ── Save manifest ──────────────────────────────────────────────────────────────
def _strip_arrays(d):
    return {k: v for k, v in d.items() if not k.startswith('y_')}

manifest = {
    'checkpoint_path': str(best_checkpoint_path),
    'roi_test_root': str(ROI_TEST_ROOT),
    'pub_test_root': str(PUB_TEST_ROOT),
    'preprocessing': 'CLAHE(1.25) -> square pad -> resize(384) -> ImageNet norm',
    'evaluation_only': True,
    'test_yolo_roi': _strip_arrays(test_roi_m),
    'test_published': _strip_arrays(test_pub_m),
    'gradcam_dir': str(CAM_DIR),
}
with open(CAM_DIR / 'evaluation_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)
print(f'Manifest: {CAM_DIR / "evaluation_manifest.json"}')


---

## What do "Published" and "YOLO-ROI" mean?

| View | Source | Description |
|---|---|---|
| **Published** | `PUB_TEST_ROOT = kneeKL224/test/` | Original full-size X-ray as it appears in the Mendeley dataset. No preprocessing applied. |
| **YOLO-ROI** | `ROI_TEST_ROOT = densenet121_yolo_square_roi_trainvaltest_v2/test/` | Your own YOLO detector cropped the knee region before feeding it to the model. Removes background, focuses on joint space. |

### Why evaluate both?

- **Published** tells you how the model behaves on the raw, unmodified images your app will receive in production.
- **YOLO-ROI** tells you how much the detector helps — if YOLO-ROI is much better, it means the detector is doing useful work.

> **Note:** `ROI_TEST_ROOT` is the `/test` subfolder of your `densenet121_yolo_square_roi_trainvaltest_v2` folder. Make sure you have a `/test` subfolder there (generated by the YOLO training split script). If that folder does not exist, re-run the YOLO data-split script to create the train/val/test split before running this notebook.


## 5. Save History & Metadata


In [11]:
pd.DataFrame(history).to_csv(RUN_DIR / 'history.csv', index=False)

metadata = {
    'architecture': 'densenet121_yolo_roi',
    'loss': 'cross_entropy',
    'epochs': EPOCHS,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'batch_size': BATCH_SIZE,
    'num_workers': NUM_WORKERS,
    'persistent_workers': PERSISTENT_WORKERS,
    'num_workers': NUM_WORKERS,
    'input_size': INPUT_SIZE,
    'alternate_view_probability': ALTERNATE_VIEW_PROBABILITY,
    'scheduler': 'cosine_annealing',
    'best_robust_selection': best_score,
    'base_checkpoint': str(BASE_CHECKPOINT),
    'published_root': str(PUBLISHED_ROOT),
    'roi_root': str(ROI_ROOT),
    'train_samples': len(train_frame),
    'val_samples': len(val_frame),
}
with open(RUN_DIR / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Saved:')
print(f'  {RUN_DIR / "best_model.pth"}')
print(f'  {RUN_DIR / "last_model.pth"}')
print(f'  {RUN_DIR / "history.csv"}')
print(f'  {RUN_DIR / "metadata.json"}')
print(f'\nBest robust_selection: {best_score:.4f}')
print(f'Next: run evaluation notebook')


Saved:
  /content/drive/MyDrive/Models/densenet121_yolo_roi/2026-08-21_03-41-45_618715_UTC/best_model.pth
  /content/drive/MyDrive/Models/densenet121_yolo_roi/2026-08-21_03-41-45_618715_UTC/last_model.pth
  /content/drive/MyDrive/Models/densenet121_yolo_roi/2026-08-21_03-41-45_618715_UTC/history.csv
  /content/drive/MyDrive/Models/densenet121_yolo_roi/2026-08-21_03-41-45_618715_UTC/metadata.json

Best robust_selection: 0.7437
Next: run evaluation notebook
